# 🎓 Exploración de Titulaciones — UJI 2010-2020

## 🎯 Qué hace
Extrae todas las titulaciones únicas presentes en `df_alumno.parquet` para identificar qué grados, licenciaturas y diplomaturas están en el dataset del TFM.

## 📋 Requisitos
- `df_alumno.parquet` en `data/01_raw/` (o ruta equivalente)
- Entorno conda `tfm_abandono` activo

## 📤 Genera
- Listado completo de titulaciones únicas (consola)
- Tabla con nº de alumnos, cursos en los que aparece, % abandono por titulación
- Excel exportado a `data/01_raw/titulaciones_inventario.xlsx`

## 🔄 Flujo
1. Carga `df_alumno.parquet`
2. Inspecciona las columnas relacionadas con titulación
3. Lista titulaciones únicas con métricas básicas
4. Detecta licenciaturas/diplomaturas (planes pre-Bolonia) vs grados
5. Busca palabras clave (agroalimentaria, agraria, etc.)
6. Exporta inventario a Excel

## ➡️ Siguiente
Compartir el listado para emparejar desaparecidas con sus equivalentes actuales.

## 1. Setup y carga

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Localizar ROOT del proyecto (busca la carpeta src/ subiendo niveles)
ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"ROOT detectado: {ROOT}")

# Configuración pandas para mejor visualización
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

ROOT detectado: c:\FF\AU_UJI_v2


In [2]:
# Carga del dataset principal
# Ajustar ruta si está en otra ubicación
RUTAS_POSIBLES = [
    ROOT / 'data' / '01_raw' / 'df_alumno.parquet',
    ROOT / 'data' / '00_ingesta' / 'df_alumno.parquet',
    ROOT / 'data' / 'df_alumno.parquet',
]

df = None
for ruta in RUTAS_POSIBLES:
    if ruta.exists():
        df = pd.read_parquet(ruta)
        print(f"✅ Cargado: {ruta}")
        print(f"   Forma: {df.shape[0]:,} filas × {df.shape[1]} columnas")
        break

if df is None:
    print("❌ df_alumno.parquet no encontrado en rutas habituales.")
    print("   Edita RUTAS_POSIBLES con la ruta correcta.")

❌ df_alumno.parquet no encontrado en rutas habituales.
   Edita RUTAS_POSIBLES con la ruta correcta.


## 2. Localizar columnas de titulación

In [3]:
# Carga del dataset principal
RUTAS_POSIBLES = [
    ROOT / 'data' / '02_processed' / 'df_alumno.parquet',  # ← AÑADIR ESTA PRIMERO
    ROOT / 'data' / '01_raw' / 'df_alumno.parquet',
    ROOT / 'data' / '00_ingesta' / 'df_alumno.parquet',
    ROOT / 'data' / 'df_alumno.parquet',
]

df = None
for ruta in RUTAS_POSIBLES:
    if ruta.exists():
        df = pd.read_parquet(ruta)
        print(f"✅ Cargado: {ruta}")
        print(f"   Forma: {df.shape[0]:,} filas × {df.shape[1]} columnas")
        break

if df is None:
    print("❌ df_alumno.parquet no encontrado en rutas habituales.")
    print("   Edita RUTAS_POSIBLES con la ruta correcta.")

✅ Cargado: c:\FF\AU_UJI_v2\data\02_processed\df_alumno.parquet
   Forma: 109,568 filas × 37 columnas


## 3. Listado de titulaciones únicas

**👇 Ajustar `COL_TITULACION` con el nombre correcto de la columna que contiene el nombre del grado.**

In [4]:
# AJUSTAR según el resultado de la celda anterior
COL_TITULACION = 'titulacion'  # ← cambiar si la columna se llama distinto
COL_CURSO = 'curso_aca_ini' if 'curso_aca_ini' in df.columns else None
COL_ABANDONO = 'abandono' if 'abandono' in df.columns else None

# Verificación
if COL_TITULACION not in df.columns:
    print(f"❌ La columna '{COL_TITULACION}' no existe. Cambia COL_TITULACION arriba.")
    print(f"   Columnas disponibles parecidas: {cols_relevantes}")
else:
    n_unicos = df[COL_TITULACION].nunique()
    print(f"✅ Columna '{COL_TITULACION}' OK")
    print(f"   Número de titulaciones únicas: {n_unicos}")
    print(f"   Columna curso: {COL_CURSO}")
    print(f"   Columna abandono: {COL_ABANDONO}")

✅ Columna 'titulacion' OK
   Número de titulaciones únicas: 40
   Columna curso: curso_aca_ini
   Columna abandono: None


In [5]:
# Listado simple ordenado alfabéticamente
titulaciones = sorted(df[COL_TITULACION].dropna().unique())

print(f"📋 LISTADO DE LAS {len(titulaciones)} TITULACIONES ÚNICAS")
print("=" * 80)
for i, t in enumerate(titulaciones, 1):
    print(f"{i:>3d}. {t}")

📋 LISTADO DE LAS 40 TITULACIONES ÚNICAS
  1. Doble Grado en Administración y Dirección de Empresas y Derecho, 
  2. Grado en Administración de Empresas
  3. Grado en Arquitectura Técnica
  4. Grado en Arquitectura Técnica (Plan 2020)
  5. Grado en Comunicación Audiovisual
  6. Grado en Criminologia y Seguridad
  7. Grado en Criminologia y Seguridad  (Plan 2020)
  8. Grado en Derecho
  9. Grado en Diseño y Desarrollo de Videojuegos
 10. Grado en Economía
 11. Grado en Enfermería
 12. Grado en Estudios Ingleses
 13. Grado en Finanzas y Contabilidad
 14. Grado en Gestión y Administración Pública
 15. Grado en Historia y Patrimonio
 16. Grado en Historia y Patrimonio (Plan 2015)
 17. Grado en Humanidades: Estudios Interculturales
 18. Grado en Ingeniería Agroalimentaria y del Medio Rural
 19. Grado en Ingeniería Agroalimentaria y del Medio Rural (Plan 2018)
 20. Grado en Ingeniería Eléctrica
 21. Grado en Ingeniería Informática
 22. Grado en Ingeniería Mecanica
 23. Grado en Ingeniería Quí

## 4. Inventario detallado por titulación

In [6]:
# Construir tabla con métricas por titulación
agg_dict = {COL_TITULACION: 'count'}  # nº alumnos

if COL_CURSO is not None:
    agg_dict[COL_CURSO] = ['min', 'max', 'nunique']

if COL_ABANDONO is not None:
    agg_dict[COL_ABANDONO] = ['sum', 'mean']

inventario = df.groupby(COL_TITULACION).agg(agg_dict)

# Aplanar columnas multi-nivel
inventario.columns = ['_'.join(c).strip('_') for c in inventario.columns]
inventario = inventario.reset_index()

# Renombrar para claridad
rename_map = {f'{COL_TITULACION}_count': 'n_alumnos'}
if COL_CURSO is not None:
    rename_map.update({
        f'{COL_CURSO}_min': 'curso_primero',
        f'{COL_CURSO}_max': 'curso_ultimo',
        f'{COL_CURSO}_nunique': 'n_cursos',
    })
if COL_ABANDONO is not None:
    rename_map.update({
        f'{COL_ABANDONO}_sum': 'n_abandonos',
        f'{COL_ABANDONO}_mean': 'tasa_abandono',
    })

inventario = inventario.rename(columns=rename_map)

# Ordenar por nº de alumnos descendente
inventario = inventario.sort_values('n_alumnos', ascending=False).reset_index(drop=True)

# Mostrar tabla
print(f"📊 INVENTARIO DETALLADO ({len(inventario)} titulaciones)")
print("=" * 80)
inventario

📊 INVENTARIO DETALLADO (40 titulaciones)


,titulacion,n_alumnos,curso_primero,curso_ultimo,n_cursos
0,Grado en Maestro en Educación Primaria,8558,2010,2018,9
1,Grado en Psicología,6968,2010,2020,11
2,Grado en Administración de Empresas,6697,2010,2020,11
3,Grado en Derecho,6678,2010,2020,11
4,Grado en Maestro en Educación Infantil,6081,2010,2018,9
5,Grado en Finanzas y Contabilidad,5095,2010,2020,11
6,Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos,4913,2010,2020,11
7,Grado en Traducción e Interpretación,4137,2009,2020,12
8,Grado en Publicidad y Relaciones Públicas,4135,2009,2020,12
9,Grado en Comunicación Audiovisual,3857,2009,2020,12


## 5. Detección de planes pre-Bolonia (licenciaturas, diplomaturas, ingenierías técnicas)

In [7]:
PALABRAS_PRE_BOLONIA = ['licenciatura', 'licenciado', 'diplomatura', 'diplomado',
                         'ingeniería técnica', 'ingeniero técnico', 'arquitectura técnica',
                         'maestro', 'mestre']
PALABRAS_GRADO = ['grado', 'grau']

def clasificar(nombre):
    n = str(nombre).lower()
    if any(p in n for p in PALABRAS_PRE_BOLONIA):
        return 'Pre-Bolonia'
    if any(p in n for p in PALABRAS_GRADO):
        return 'Grado (Bolonia)'
    return 'Sin clasificar'

inventario['plan_estudios'] = inventario[COL_TITULACION].apply(clasificar)

# Resumen
print("🏛️ Distribución por tipo de plan de estudios:")
print(inventario['plan_estudios'].value_counts())
print()
print("📋 Detalle:")
for tipo in inventario['plan_estudios'].unique():
    print(f"\n--- {tipo} ---")
    for nombre in inventario[inventario['plan_estudios'] == tipo][COL_TITULACION]:
        print(f"   · {nombre}")

🏛️ Distribución por tipo de plan de estudios:
plan_estudios
Grado (Bolonia)    34
Pre-Bolonia         6
Name: count, dtype: int64

📋 Detalle:

--- Pre-Bolonia ---
   · Grado en Maestro en Educación Primaria
   · Grado en Maestro en Educación Infantil
   · Grado en Arquitectura Técnica
   · Grado en Maestro en Educación Primaria (Plan 2018)
   · Grado en Maestro en Educación Infantil (Plan 2018)
   · Grado en Arquitectura Técnica (Plan 2020)

--- Grado (Bolonia) ---
   · Grado en Psicología
   · Grado en Administración de Empresas
   · Grado en Derecho
   · Grado en Finanzas y Contabilidad
   · Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos
   · Grado en Traducción e Interpretación
   · Grado en Publicidad y Relaciones Públicas
   · Grado en Comunicación Audiovisual
   · Grado en Periodismo
   · Grado en Criminologia y Seguridad
   · Grado en Ingeniería Informática
   · Grado en Ingeniería Mecanica
   · Grado en Relaciones Laborales y Recursos Humanos
   · Grado en M

## 6. Búsqueda de palabras clave (agro, ingeniería, etc.)

In [8]:
# Buscar titulaciones por palabras clave de interés
BUSQUEDAS = {
    '🌾 Agro/agraria/rural': ['agro', 'agra', 'rural', 'aliment'],
    '⚙️ Ingenierías': ['ingenier', 'enginyer'],
    '🏥 Salud': ['enfermer', 'medicin', 'psicolog', 'fisioterap'],
    '⚖️ Jurídicas': ['derecho', 'criminolog', 'relaciones laboral'],
    '💼 Económicas': ['economía', 'finanz', 'empresa', 'admin', 'marketing', 'turismo'],
    '📚 Humanidades': ['filolog', 'humanid', 'historia', 'tradu', 'inglés'],
    '📰 Comunicación': ['periodism', 'audiovis', 'publicid'],
    '👶 Magisterio': ['maestro', 'mestre', 'magist'],
    '🔬 Ciencias': ['química', 'física', 'matem', 'biolog', 'ambiental'],
    '🎨 Otros': ['diseño', 'videojuego', 'deport'],
}

for categoria, palabras in BUSQUEDAS.items():
    coincidencias = []
    for nombre in titulaciones:
        n = str(nombre).lower()
        if any(p in n for p in palabras):
            coincidencias.append(nombre)
    if coincidencias:
        print(f"\n{categoria} ({len(coincidencias)}):")
        for c in coincidencias:
            print(f"   · {c}")
    else:
        print(f"\n{categoria}: ❌ ninguna coincidencia")


🌾 Agro/agraria/rural (2):
   · Grado en Ingeniería Agroalimentaria y del Medio Rural
   · Grado en Ingeniería Agroalimentaria y del Medio Rural (Plan 2018)

⚙️ Ingenierías (9):
   · Grado en Ingeniería Agroalimentaria y del Medio Rural
   · Grado en Ingeniería Agroalimentaria y del Medio Rural (Plan 2018)
   · Grado en Ingeniería Eléctrica
   · Grado en Ingeniería Informática
   · Grado en Ingeniería Mecanica
   · Grado en Ingeniería Química
   · Grado en Ingeniería de la Edificación
   · Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos
   · Grado en Ingeniería en Tecnologías Industriales

🏥 Salud (4):
   · Grado en Enfermería
   · Grado en Medicina
   · Grado en Medicina (Plan 2017)
   · Grado en Psicología

⚖️ Jurídicas (5):
   · Doble Grado en Administración y Dirección de Empresas y Derecho, 
   · Grado en Criminologia y Seguridad
   · Grado en Criminologia y Seguridad  (Plan 2020)
   · Grado en Derecho
   · Grado en Relaciones Laborales y Recursos Humanos

💼 Eco

## 7. Exportar inventario a Excel

In [9]:
ruta_salida = ROOT / 'data' / '01_raw' / 'titulaciones_inventario.xlsx'
ruta_salida.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
    inventario.to_excel(writer, sheet_name='Inventario', index=False)
    
    # Hoja con listado simple
    pd.DataFrame({
        '#': range(1, len(titulaciones) + 1),
        'titulacion': titulaciones
    }).to_excel(writer, sheet_name='Listado', index=False)

print(f"✅ Inventario guardado en: {ruta_salida}")
print(f"   Hojas: 'Inventario' (con métricas) + 'Listado' (simple)")

✅ Inventario guardado en: c:\FF\AU_UJI_v2\data\01_raw\titulaciones_inventario.xlsx
   Hojas: 'Inventario' (con métricas) + 'Listado' (simple)


## 8. Resumen final

Pasar a Claude:
1. La salida de la celda 3 (listado completo numerado)
2. La salida de la celda 6 (búsqueda por categorías) — especialmente si **no hay agro**

Con eso se puede:
- Confirmar si el grado de Agroalimentaria está o no
- Identificar licenciaturas/diplomaturas extinguidas y emparejarlas con sus grados equivalentes
- Detectar grados que faltan respecto a la oferta actual de la UJI

In [10]:
# Resumen ejecutivo
print("=" * 80)
print("📊 RESUMEN EJECUTIVO")
print("=" * 80)
print(f"Total titulaciones únicas: {len(titulaciones)}")
if 'plan_estudios' in inventario.columns:
    for tipo, n in inventario['plan_estudios'].value_counts().items():
        print(f"   · {tipo}: {n}")
if COL_CURSO is not None:
    print(f"\nRango de cursos: {df[COL_CURSO].min()} → {df[COL_CURSO].max()}")
print(f"\nTotal alumnos en dataset: {len(df):,}")
if COL_ABANDONO is not None:
    tasa = df[COL_ABANDONO].mean() * 100
    print(f"Tasa abandono global: {tasa:.2f}%")

📊 RESUMEN EJECUTIVO
Total titulaciones únicas: 40
   · Grado (Bolonia): 34
   · Pre-Bolonia: 6

Rango de cursos: 2009 → 2020

Total alumnos en dataset: 109,568
